In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [2]:
# Load and preprocess dataset
def load_text_data(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        text = f.read()
    return text.split('\n')  # Split into sentences


In [3]:
# Tokenization
def tokenize_text(sentences, vocab_size=5000):
    tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
    tokenizer.fit_on_texts(sentences)
    return tokenizer

In [4]:
# Create input-output sequences
def create_sequences(tokenizer, sentences, max_len=30):
    input_sequences = []
    for sentence in sentences:
        token_list = tokenizer.texts_to_sequences([sentence])[0]
        for i in range(1, len(token_list)):
            input_sequences.append(token_list[:i+1])
    return pad_sequences(input_sequences, maxlen=max_len, padding='pre')

In [5]:
# Load data
sentences = load_text_data("1661-0.txt")
tokenizer = tokenize_text(sentences)
padded_sequences = create_sequences(tokenizer, sentences)

In [6]:
# Prepare X and y
X, y = padded_sequences[:, :-1], padded_sequences[:, -1]
y = tf.keras.utils.to_categorical(y, num_classes=len(tokenizer.word_index) + 1)

In [7]:
# Optimized LSTM Model
def create_model(vocab_size, max_len):
    model = Sequential([
        Embedding(vocab_size, 64, input_length=max_len-1),
        LSTM(128, return_sequences=True),
        Dropout(0.2),
        LSTM(64),
        Dense(64, activation='relu'),
        Dense(vocab_size, activation='softmax')
    ])
    model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])
    return model

In [73]:
# Improved callbacks
def get_callbacks():
    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=15,  # Increased patience
        restore_best_weights=True,
        min_delta=0.001  # Minimum change to qualify as an improvement
    )
    
    reduce_lr = ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=5,
        min_lr=0.00001,
        verbose=1
    )
    
    return [early_stop, reduce_lr]

In [74]:
# Improved training function
def train_model(model, X, y, batch_size=32, epochs=50):
    # Split data with less validation
    validation_split = 0.1  # Reduced from 0.2
    
    # Get callbacks
    callbacks = get_callbacks()
    
    # Train with class weights if needed
    history = model.fit(
        X, y,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=validation_split,
        callbacks=callbacks,
        shuffle=True
    )
    
    return history

In [ ]:
# Create and train model
vocab_size = len(tokenizer.word_index) + 1
model = create_model(vocab_size, max_len=30)
model.fit(X, y, epochs=30, batch_size=32, validation_split=0.1)

In [78]:
# Prediction function
def predict_next_words(model, tokenizer, seed_text, max_len, num_words=10):
    for _ in range(num_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_len - 1, padding='pre')
        predicted_index = np.argmax(model.predict(token_list, verbose=0))
        predicted_word = tokenizer.index_word.get(predicted_index, "")
        if not predicted_word:
            break
        seed_text += " " + predicted_word
    return seed_text

In [ ]:
# Example Usage
seed_text = "The Adventures"
generated_text = predict_next_words(model, tokenizer, seed_text, max_len=30, num_words=20)
print("Generated Text:", generated_text)